In [8]:
### 1.Работа с API. Извлечение данных изи OpenAPI T-Invest.
import os
import re
import requests
import logging
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

load_dotenv()

TOKEN = os.getenv('T_TOKEN')
POSTGRES_USER = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB = os.getenv('POSTGRES_DB')
URL = 'https://invest-public-api.tbank.ru/rest/tinkoff.public.invest.api.contract.v1.InstrumentsService/Currencies'

def camel_to_snake(name):
    """Преобразует camelCase в snake_case"""
    # Вставляем _ перед заглавными буквами (кроме первой)
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)

    return name.lower()



headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json"
    }

data = {
  "instrumentStatus": "INSTRUMENT_STATUS_ALL",
  "instrumentExchange": "INSTRUMENT_EXCHANGE_DEALER"
}

response = requests.post(url = URL, headers=headers, json=data, verify=False)

if response.status_code == 200:
    # return response.json()
    df = pd.json_normalize(response.json()['instruments'], sep='_')
    df.head()
else:
    print(f"Ошибка: {response.status_code}")
    print(response.text)
    # return None
    print(None)

# Применяем ко всем столбцам
df.columns = [camel_to_snake(col) for col in df.columns]

/home/kam/datalearnhome/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'invest-public-api.tbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [9]:
df.shape

(29, 48)

In [10]:
df[df['figi'] == 'BBG004730N88']

,figi,ticker,class_code,isin,lot,currency,short_enabled_flag,name,exchange,country_of_risk,...,dshort_units,dshort_nano,dlong_min_units,dlong_min_nano,dshort_min_units,dshort_min_nano,dlong_client_units,dlong_client_nano,dshort_client_units,dshort_client_nano
